# Robustness Checks

## Setup

In [ ]:
import os
import pickle
import sys
import warnings
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

from config import BATCH_SIZE, SEED
from src.dataloader import get_dataloaders
from src.model import DissagreementPredictor
from src.utils import set_seed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIGURES_DIR = PROJECT_ROOT / "figures"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
RAW_DIR = PROJECT_ROOT / "data/raw"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], dtype=torch.float32).view(1, 3, 1, 1)
CIFAR10_STD = torch.tensor([0.2023, 0.1994, 0.2010], dtype=torch.float32).view(1, 3, 1, 1)

set_seed(SEED)
DEVICE

In [ ]:
def load_cifar_batch(batch_path):
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"dtype\(\): align should be passed as Python or NumPy boolean.*",
            category=np.exceptions.VisibleDeprecationWarning,
        )
        with open(batch_path, "rb") as f:
            batch = pickle.load(f, encoding="bytes")
    return batch


def resolve_checkpoint_path(model_path):
    model_path = Path(model_path)
    if model_path.exists():
        return model_path

    candidate_names = [
        model_path.name,
        model_path.name.replace("best_model_", "best_model _"),
        model_path.name.replace(" ", ""),
    ]

    for candidate_name in candidate_names:
        candidate_path = model_path.with_name(candidate_name)
        if candidate_path.exists():
            return candidate_path

    raise FileNotFoundError(
        f"Could not find checkpoint for {model_path.name}. Available files: "
        f"{sorted(path.name for path in model_path.parent.glob('*'))}"
    )


def compute_distribution_metrics(all_preds, all_true, eps=1e-8):
    preds = np.clip(all_preds, eps, 1.0)
    true = np.clip(all_true, eps, 1.0)

    kl_values = np.sum(true * (np.log(true) - np.log(preds)), axis=1)
    m = 0.5 * (true + preds)
    jsd_values = 0.5 * np.sum(true * (np.log(true) - np.log(m)), axis=1)
    jsd_values += 0.5 * np.sum(preds * (np.log(preds) - np.log(m)), axis=1)

    return {
        "KL_mean": float(np.mean(kl_values)),
        "JSD_mean": float(np.mean(jsd_values)),
    }


def collect_predictions(model, data_loader):
    all_preds = []
    all_true = []
    model.eval()
    with torch.no_grad():
        for imgs, targets in data_loader:
            preds = model(imgs.to(DEVICE))
            all_preds.append(preds.cpu().numpy())
            all_true.append(targets.numpy())
    return np.concatenate(all_preds, axis=0), np.concatenate(all_true, axis=0)


def ensure_cifar10h_raw():
    raw_path = RAW_DIR / "cifar10h-raw.npy"
    if raw_path.exists():
        return raw_path

    zip_path = RAW_DIR / "cifar10h-raw.zip"
    url = "https://github.com/jcpeterson/cifar-10h/raw/master/data/cifar10h-raw.zip"
    print(f"Downloading {url}")
    urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(RAW_DIR)

    if not raw_path.exists():
        raise FileNotFoundError("cifar10h-raw.npy was not found after extraction.")
    return raw_path


batch = load_cifar_batch(PROJECT_ROOT / "data/raw/cifar-10-batches-py/test_batch")
images = batch[b"data"]
hard_labels_all = np.array(batch[b"labels"])
probs = np.load(PROJECT_ROOT / "data/raw/cifar10h-probs.npy")
test_idx = np.load(PROJECT_ROOT / "data/processed/test_idx.npy")

_, _, test_loader = get_dataloaders(images, probs, batch_size=BATCH_SIZE)
hard_labels = hard_labels_all[test_idx]

model = DissagreementPredictor().to(DEVICE)
model.load_state_dict(torch.load(resolve_checkpoint_path(CHECKPOINTS_DIR / "best_model_kl.pt"), map_location=DEVICE))

all_preds, all_true = collect_predictions(model, test_loader)
all_preds.shape, all_true.shape, hard_labels.shape

## Check A — Annotator Subsampling

In [ ]:
raw_path = ensure_cifar10h_raw()
cifar10h_raw = np.load(raw_path)
cifar10h_raw_test = cifar10h_raw[test_idx]


def compute_jsd_batch(p, q, eps=1e-8):
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    m = 0.5 * (p + q)
    jsd = 0.5 * np.sum(p * (np.log(p) - np.log(m)), axis=1)
    jsd += 0.5 * np.sum(q * (np.log(q) - np.log(m)), axis=1)
    return jsd


def subsample_distribution(responses, k, rng):
    valid = responses[responses >= 0].astype(int)
    sample_size = min(k, len(valid))
    chosen = rng.choice(valid, size=sample_size, replace=False)
    counts = np.bincount(chosen, minlength=10).astype(np.float32)
    return counts / counts.sum()


rng = np.random.default_rng(SEED)
k_values = [5, 10, 20, 35, 51]
subsampling_rows = []

for k in k_values:
    subsampled = np.stack([subsample_distribution(row, k, rng) for row in cifar10h_raw_test], axis=0)
    jsd_sub_vs_full = compute_jsd_batch(subsampled, all_true).mean()
    jsd_model_vs_sub = compute_jsd_batch(all_preds, subsampled).mean()
    subsampling_rows.append({
        "k": k,
        "JSD_subsample_vs_full": float(jsd_sub_vs_full),
        "JSD_model_vs_subsample": float(jsd_model_vs_sub),
    })
    print(subsampling_rows[-1])

subsampling_rows

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot([row["k"] for row in subsampling_rows], [row["JSD_subsample_vs_full"] for row in subsampling_rows], marker="o", linewidth=2, label="Subsample vs Full")
plt.plot([row["k"] for row in subsampling_rows], [row["JSD_model_vs_subsample"] for row in subsampling_rows], marker="o", linewidth=2, label="Model vs Subsample")
plt.xlabel("Number of Annotators (k)")
plt.ylabel("JSD")
plt.title("Annotator Subsampling Robustness")
plt.xticks(k_values)
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "annotator_subsampling.png", dpi=150, bbox_inches="tight")
plt.show()

## Check B — OOD Corruption

In [ ]:
def denormalize(x):
    mean = CIFAR10_MEAN.to(x.device)
    std = CIFAR10_STD.to(x.device)
    return x * std + mean


def normalize(x):
    mean = CIFAR10_MEAN.to(x.device)
    std = CIFAR10_STD.to(x.device)
    return (x - mean) / std


def gaussian_noise(x, severity):
    noise_std = [0.04, 0.08, 0.12, 0.18, 0.26][severity - 1]
    x_denorm = denormalize(x)
    x_noisy = torch.clamp(x_denorm + torch.randn_like(x_denorm) * noise_std, 0.0, 1.0)
    return normalize(x_noisy)


def gaussian_kernel(kernel_size, sigma, device):
    coords = torch.arange(kernel_size, dtype=torch.float32, device=device) - (kernel_size - 1) / 2
    yy, xx = torch.meshgrid(coords, coords, indexing="ij")
    kernel = torch.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    kernel = kernel / kernel.sum()
    return kernel.view(1, 1, kernel_size, kernel_size)


def gaussian_blur(x, severity):
    kernel_size = [3, 5, 7, 9, 11][severity - 1]
    sigma = [0.5, 0.8, 1.2, 1.6, 2.0][severity - 1]
    x_denorm = denormalize(x)
    kernel = gaussian_kernel(kernel_size, sigma, x.device).repeat(x.size(1), 1, 1, 1)
    x_blur = F.conv2d(x_denorm, kernel, padding=kernel_size // 2, groups=x.size(1))
    return normalize(torch.clamp(x_blur, 0.0, 1.0))


def contrast(x, severity):
    factor = [0.85, 0.7, 0.55, 0.4, 0.25][severity - 1]
    x_denorm = denormalize(x)
    mean = x_denorm.mean(dim=(2, 3), keepdim=True)
    x_contrast = torch.clamp((x_denorm - mean) * factor + mean, 0.0, 1.0)
    return normalize(x_contrast)


def mean_predicted_entropy(preds, eps=1e-8):
    preds = np.clip(preds, eps, 1.0)
    entropy_values = -np.sum(preds * np.log2(preds), axis=1)
    return float(np.mean(entropy_values))


corruptions = {
    "gaussian_noise": gaussian_noise,
    "gaussian_blur": gaussian_blur,
    "contrast": contrast,
}

subset_images = []
seen = 0
for imgs, _ in test_loader:
    remaining = 1000 - seen
    if remaining <= 0:
        break
    subset_images.append(imgs[:remaining])
    seen += min(remaining, imgs.size(0))

subset_images = torch.cat(subset_images, dim=0).to(DEVICE)
severity_levels = [1, 2, 3, 4, 5]
robustness_results = {name: [] for name in corruptions}

model.eval()
with torch.no_grad():
    for corruption_name, corruption_fn in corruptions.items():
        for severity in severity_levels:
            corrupted = corruption_fn(subset_images, severity)
            preds = model(corrupted).cpu().numpy()
            entropy_mean = mean_predicted_entropy(preds)
            robustness_results[corruption_name].append(entropy_mean)
            print(corruption_name, severity, entropy_mean)

In [ ]:
plt.figure(figsize=(7, 4.5))
for corruption_name, entropy_values in robustness_results.items():
    plt.plot(severity_levels, entropy_values, marker="o", linewidth=2, label=corruption_name)
plt.xlabel("Severity")
plt.ylabel("Mean Predicted Entropy")
plt.title("OOD Corruption Robustness")
plt.xticks(severity_levels)
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "robustness_entropy.png", dpi=150, bbox_inches="tight")
plt.show()

## Check C — Class-conditional Performance

In [ ]:
classwise_rows = []
for class_idx, class_name in enumerate(CLASS_NAMES):
    mask = hard_labels == class_idx
    metrics = compute_distribution_metrics(all_preds[mask], all_true[mask])
    row = {
        "class_idx": class_idx,
        "class_name": class_name,
        "KL_mean": metrics["KL_mean"],
        "JSD_mean": metrics["JSD_mean"],
    }
    classwise_rows.append(row)
    print(row)

classwise_rows

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh([row["class_name"] for row in classwise_rows], [row["JSD_mean"] for row in classwise_rows], color="#4C78A8")
plt.xlabel("JSD")
plt.ylabel("Class")
plt.title("Class-conditional JSD")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "classwise_jsd.png", dpi=150, bbox_inches="tight")
plt.show()